# 1. The Anveshar pipeline harness

A reproducible, provenance-tracked pipeline that turns a rare cancer into a cited, confidence-scored cross-condition translation dossier. Five auditable stages: resolve, databases, target validation, translate, assemble. Offline is deterministic; `live=True` queries Open Targets, Ensembl, ChEMBL, PubMed, and DepMap.

In [1]:
import os, sys
def _find_repo():
    p = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(p, "anveshar", "__init__.py")):
            return p
        p = os.path.dirname(p)
    return None
REPO = _find_repo()
if REPO and REPO not in sys.path:
    sys.path.insert(0, REPO)
import anveshar
print("anveshar loaded from", os.path.dirname(anveshar.__file__))

anveshar loaded from /data1/lesliec/vijay/github/anveshar/anveshar


## Run the harness offline (deterministic)

In [2]:
from anveshar.pipeline import run
d = run("renal medullary carcinoma", live=False)
print(d["condition"], "| drivers:", d["drivers"], "| mode:", d["mode"])
s = d["summary"]; bc = s["best_confidence"]
print("actionable:", s["n_actionable"], "| tissue-agnostic:", s["n_tissue_agnostic"],
      "| cross-condition:", s["n_cross_condition"])
print("best confidence:", bc)

Renal medullary carcinoma | drivers: ['SMARCB1'] | mode: offline
actionable: 1 | tissue-agnostic: 0 | cross-condition: 8
best confidence: {'label': 'Moderate', 'pct': 68, 'basis': 'approved in another condition on a shared target, or a prospective basket signal, with a linked citation'}


## Provenance: every stage is auditable

In [3]:
import pandas as pd
pd.DataFrame(d["provenance"])

,stage,source,query,n,note
0,resolve,Anveshar catalog,renal medullary carcinoma,1,matched Renal medullary carcinoma; drivers SMA...
1,databases,"Open Targets, Ensembl, ChEMBL",SMARCB1,0,offline mode: live database stage skipped
2,validation,"Open Targets tractability, DepMap essentiality","SMARCB1, EZH2",0,offline mode: target validation skipped
3,translate,Anveshar actionable driver map,SMARCB1,1,"1 borrowable approved therapies, 8 shared driv..."
4,workflow,Anveshar workflow generator,Renal medullary carcinoma,7,loss_or_epigenetic class; 7 stage comp-bio wor...


## Borrowable approved therapies (cited, confidence-scored)

In [4]:
pd.DataFrame([{"gene": m["gene"], "drug": m["drug"], "approved_in": m["approved_in"],
               "tier": m["tier"], "citation": m["citation"].get("url", "")}
              for m in d["translation"]["actionable"]])

,gene,drug,approved_in,tier,citation
0,SMARCB1,Tazemetostat,INI1 (SMARCB1) negative epithelioid sarcoma,2,https://pubmed.ncbi.nlm.nih.gov/33035459/


## The bespoke, driver-class-tailored comp-bio workflow

In [5]:
w = d["workflow"]
print("driver class:", w["driver_class"], "\n")
for i, st in enumerate(w["steps"], 1):
    print(str(i) + ". " + st["title"])
    print("   tools:", st["tools"])
    print("   checkpoint:", st["outputs"])
    print()

driver class: loss_or_epigenetic 

1. Confirm the loss and target the induced EZH2 dependency
   tools: DepMap co-dependency, Open Targets tractability of EZH2, CRISPR knockdown
   checkpoint: validated driver and a selective dependency; SMARCB1 loss is confirmed and cells depend on EZH2; EZH2 tractability supports drugging it where SMARCB1 itself is a common essential gene that cannot be targeted selectively.

2. Map the cross condition evidence
   tools: PubMed, Consensus, Paperclip full text, ClinicalTrials.gov; deep-research skill
   checkpoint: a graded evidence table of prior responses and resistance, with citations

3. Interpret the patient's specific alteration
   tools: OncoKB, ClinVar, AlphaMissense, Evo 2; anveshar.variants
   checkpoint: an actionability tier and a defensible call for any uncertain variant

4. Profile expression and the microenvironment
   tools: GEO, TCGA, cBioPortal, scanpy; single-cell-rna-qc and scvi-tools skills
   checkpoint: target expression, cell o

## Induced dependency (synthetic lethality) for loss drivers

In [6]:
d.get("induced_dependencies", {})

{'SMARCB1': {'dependency': 'EZH2',
  'relationship': 'SWI/SNF (SMARCB1) loss shifts chromatin control to the PRC2/EZH2 complex, creating an EZH2 dependency',
  'drug': 'Tazemetostat (EZH2 inhibitor)',
  'citation': {'label': 'Gounder et al., Lancet Oncol 2020 (tazemetostat in INI1/SMARCB1 negative epithelioid sarcoma proves the EZH2 dependency)',
   'url': 'https://pubmed.ncbi.nlm.nih.gov/33035459/'}}}

## Optional: run live (Open Targets + DepMap functional genomics)
Requires internet; wrapped so a rate limit does not break the notebook.

In [7]:
try:
    dl = run("renal medullary carcinoma", live=True)
    tv = dl.get("target_validation", {})
    display(pd.DataFrame([{"gene": g, "score": v.get("score"), "verdict": v.get("verdict"),
                           "tractability_SM": v.get("sm_tractability"), "selective": v.get("selective_dependency")}
                          for g, v in tv.items() if v.get("available")]))
except Exception as e:
    print("live stage skipped:", type(e).__name__, e)

,gene,score,verdict,tractability_SM,selective
0,SMARCB1,42,"Common essential gene, selectivity risk for a ...",6,False
1,EZH2,95,"Validated, tractable target",10,True


---
*Disclaimer: this notebook produces research and educational analysis of public data, not medical advice. Confidence and validation scores summarize evidence strength, not the probability of benefit for any individual. Every clinical decision must be made by a qualified health care provider, ideally within a clinical trial.*

*Developed by Dig Vijay Kumar Yarlagadda, [digvijayky.com](https://digvijayky.com).*